In [1]:
import os
from dotenv import load_dotenv
from typing import TypedDict

from sqlalchemy import create_engine, text
from langchain_community.utilities import SQLDatabase
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

if os.environ.get("OPENAI_API_KEY"):
    print("API exists")

C:\Users\vigna\AppData\Local\Temp\ipykernel_27304\388641553.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


API exists


In [2]:
# loading env variables and setting up database + llm
load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", ".env"))

Database_URL = os.environ.get("DATABASE_URL")
engine = create_engine(Database_URL)
db = SQLDatabase(engine)

# using the API key from .env for the model
llm = ChatOpenAI(
    model="gpt-5-mini",
    temperature=0,
    api_key=os.environ.get("OPENAI_API_KEY")
)

In [3]:
# state that gets passed between nodes
class AgentState(TypedDict):
    question: str
    schema_info: str
    sql_query: str
    query_result: str
    final_answer: str

# step 1: get the database schema
def get_schema(state: AgentState):
    schema_info = db.get_table_info()
    print("got the schema")
    return {"schema_info": schema_info}

In [4]:
# step 2: send the question + schema to GPT and get a SQL query back
def generate_sql(state: AgentState):
    prompt = f"""You are a PostgreSQL expert. Based on the database schema below,
write a SQL query that answers the user's question.

Rules:
- Only return the SQL query, nothing else
- Use double quotes for column names with uppercase letters
- Use LIMIT to keep results small

Schema:
{state["schema_info"]}

Question: {state["question"]}

SQL:"""

    response = llm.invoke(prompt)
    sql_query = response.content.strip()
    print(f"generated query:\n{sql_query}")
    return {"sql_query": sql_query}

In [5]:
# step 3: run the query on the neon database
def execute_query(state: AgentState):
    try:
        with engine.connect() as conn:
            result = conn.execute(text(state["sql_query"]))
            rows = result.fetchall()
            columns = list(result.keys())

            if not rows:
                query_result = "no results found"
            else:
                header = " | ".join(columns)
                data = "\n".join(
                    " | ".join(str(val) for val in row) for row in rows
                )
                query_result = f"{header}\n{data}"

        print(f"query ran successfully, got {len(rows)} rows")
    except Exception as e:
        query_result = f"error: {str(e)}"
        print(f"query failed: {e}")

    return {"query_result": query_result}

In [6]:
# step 4: turn the raw result into a readable answer
def format_answer(state: AgentState):
    prompt = f"""Based on the question and query result below, give a short clear answer.

Question: {state["question"]}
SQL Query: {state["sql_query"]}
Result:
{state["query_result"]}

Answer:"""

    response = llm.invoke(prompt)
    return {"final_answer": response.content.strip()}

In [7]:
# building the langgraph workflow
# flow: get schema -> generate sql -> run query -> format answer
workflow = StateGraph(AgentState)

workflow.add_node("get_schema", get_schema)
workflow.add_node("generate_sql", generate_sql)
workflow.add_node("execute_query", execute_query)
workflow.add_node("format_answer", format_answer)

workflow.add_edge(START, "get_schema")
workflow.add_edge("get_schema", "generate_sql")
workflow.add_edge("generate_sql", "execute_query")
workflow.add_edge("execute_query", "format_answer")
workflow.add_edge("format_answer", END)

graph = workflow.compile()

In [8]:
res = graph.invoke({"question": "who had the most orders?"})
answer = res["final_answer"]
answer

got the schema
generated query:
SELECT c."CustomerKey", c."FirstName", c."LastName", COUNT(DISTINCT s."OrderNumber") AS order_count
FROM sales_data s
JOIN customer_lookup c ON CAST(s."CustomerKey" AS TEXT) = c."CustomerKey"
GROUP BY c."CustomerKey", c."FirstName", c."LastName"
ORDER BY order_count DESC
LIMIT 1;
query ran successfully, got 1 rows


'DALTON PEREZ (CustomerKey 11091) — 26 orders.'

In [9]:
# agent 2: takes the answer from agent 1 and displays it
class DisplayState(TypedDict):
    input_answer: str
    display_output: str

def show_answer(state: DisplayState):
    output = f"Agent 2 received: {state['input_answer']}"
    print(output)
    return {"display_output": output}

# building agent 2 graph
display_workflow = StateGraph(DisplayState)
display_workflow.add_node("show_answer", show_answer)
display_workflow.add_edge(START, "show_answer")
display_workflow.add_edge("show_answer", END)

display_graph = display_workflow.compile()

# passing the answer from agent 1 to agent 2
result = display_graph.invoke({"input_answer": answer})
result["display_output"]

Agent 2 received: DALTON PEREZ (CustomerKey 11091) — 26 orders.


'Agent 2 received: DALTON PEREZ (CustomerKey 11091) — 26 orders.'

In [10]:

# 1. First, install gradio if you haven't: !uv pip install gradio
import gradio as gr

# This is the function the website will call when a user submits a prompt
def retrieve_data(user_prompt):
    try:
        # We use the graph we already built in the notebook
        result = graph.invoke({"question": user_prompt})
        
        # Format the output beautifully
        answer = result["final_answer"]
        sql_used = result["sql_query"]
        
        return answer, sql_used
    except Exception as e:
        return f"Error: {str(e)}", "No SQL generated"

# Custom CSS for the Green & White theme
custom_theme = gr.themes.Soft(
    primary_hue="emerald",  # Green color
    secondary_hue="green",
    neutral_hue="slate",
)

# Build the Web Interface
with gr.Blocks(theme=custom_theme, title="viggy's data retrieval AI") as website:
    gr.Markdown("<h1 style='text-align: center; color: #10B981;'>🌿 viggy's data retrieval AI</h1>")
    gr.Markdown("<p style='text-align: center;'>Ask me anything about your database!</p>")
    
    with gr.Row():
        with gr.Column(scale=2):
            # Input space
            prompt_input = gr.Textbox(
                lines=3, 
                placeholder="E.g., Who had the most orders?", 
                label="Ask your question here"
            )
            submit_btn = gr.Button("Search Data", variant="primary")
            
        with gr.Column(scale=3):
            # Output space
            answer_output = gr.Textbox(label="Answer", lines=2, interactive=False)
            sql_output = gr.Code(label="SQL Query Used (For Transparency)", language="sql", interactive=False)
            
    # Connect the button to the function
    submit_btn.click(
        fn=retrieve_data, 
        inputs=[prompt_input], 
        outputs=[answer_output, sql_output]
    )

# This launches the website directly from the notebook
website.launch()






d:\agentic ai with neon\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\vigna\AppData\Local\Temp\ipykernel_27304\3756964095.py:26: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=custom_theme, title="viggy's data retrieval AI") as website:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


d:\agentic ai with neon\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


got the schema
generated query:
SELECT c."CustomerKey", c."FirstName", c."LastName", c."EmailAddress", COUNT(DISTINCT s."OrderNumber") AS "OrderCount"
FROM sales_data s
JOIN customer_lookup c ON s."CustomerKey"::text = c."CustomerKey"
GROUP BY c."CustomerKey", c."FirstName", c."LastName", c."EmailAddress"
ORDER BY "OrderCount" DESC
LIMIT 1;
query ran successfully, got 1 rows
